# Exercise 4

In [1]:
from datasets import Dataset

file = open("chamber_of_secrets.txt", "r")
raw = file.read().replace("\n", " ").strip()
file.close()

chapters = ["CHAPTER" + chapter for chapter in raw.split("CHAPTER")]
# Remove title
chapters.pop(0)

datasetDict = Dataset.from_dict({"chapters": chapters[1:]})
datasetDict = datasetDict.train_test_split(test_size=0.2)

In [2]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('distilgpt2')

def tokenize(el):
    outputs = tokenizer(
        el["chapters"],
        truncation=True,
        max_length=128,
        return_overflowing_tokens=True,
        return_length=True
    )
    inputBatch = []
    for l, ids in zip(outputs["length"], outputs["input_ids"]):
        if l == 128:
            inputBatch.append(ids)

    return {"input_ids": inputBatch}

tokenizedData = datasetDict.map(
    tokenize, batched=True, remove_columns=datasetDict["train"].column_names
)

  0%|          | 0/1 [00:00<?, ?ba/s]

  0%|          | 0/1 [00:00<?, ?ba/s]

In [4]:
from transformers import TFAutoModelForCausalLM

model = TFAutoModelForCausalLM.from_pretrained("distilgpt2")

2022-04-17 11:04:09.024672: I tensorflow/core/platform/cpu_feature_guard.cc:151] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2022-04-17 11:04:09.041221: W tensorflow/python/util/util.cc:368] Sets are not currently considered sequences, but this may change in the future, so consider avoiding using them.
All model checkpoint layers were used when initializing TFGPT2LMHeadModel.

All the layers of TFGPT2LMHeadModel were initialized from the model checkpoint at distilgpt2.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFGPT2LMHeadModel for predictions without further training.


In [5]:
from transformers import DataCollatorForLanguageModeling

tokenizer.pad_token = tokenizer.eos_token
dataCollator = DataCollatorForLanguageModeling(tokenizer, mlm=False, return_tensors="tf")

In [6]:
from transformers import AdamWeightDecay

optimizer = AdamWeightDecay(learning_rate=2e-5, weight_decay_rate=0.01)
model.compile(optimizer)

No loss specified in compile() - the model's internal loss computation will be used as the loss. Don't panic - this is a common way to train TensorFlow models in Transformers! To disable this behaviour, please pass a loss argument, or explicitly pass `loss=None` if you do not want your model to compute a loss.


In [7]:
train_set = tokenizedData["train"].to_tf_dataset(
    columns=["attention_mask", "input_ids", "labels"],
    dummy_labels=True,
    shuffle=True,
    batch_size=32,
    collate_fn=dataCollator,
)

test_set = tokenizedData["test"].to_tf_dataset(
    columns=["attention_mask", "input_ids", "labels"],
    dummy_labels=True,
    shuffle=False,
    batch_size=32,
    collate_fn=dataCollator,
)


model.fit(x=train_set, validation_data=test_set, epochs=4)


Epoch 1/4
17/17 [==============================] - 227s 13s/step - loss: 4.0312 - val_loss: 3.7363
Epoch 2/4
17/17 [==============================] - 205s 12s/step - loss: 3.8478 - val_loss: 3.6617
Epoch 3/4
17/17 [==============================] - 203s 12s/step - loss: 3.7502 - val_loss: 3.6118
Epoch 4/4
17/17 [==============================] - 209s 12s/step - loss: 3.6733 - val_loss: 3.5768


In [8]:
from transformers import pipeline
    
pipe = pipeline(
    "text-generation", model=model, tokenizer=tokenizer
)

txts=["Mr. and Mrs. Dursley, of number four, Privet Drive, were proud to say that they were perfectly",
"Mr. Dursley was the director of a firm called",
"The Dursleys had everything they wanted, but"]

for txt in txts:
    print(pipe(txt, num_return_sequences=1)[0]["generated_text"])


Setting `pad_token_id` to 50256 (first `eos_token_id`) to generate sequence
Setting `pad_token_id` to 50256 (first `eos_token_id`) to generate sequence


Mr. and Mrs. Dursley, of number four, Privet Drive, were proud to say that they were perfectly prepared all evening, and they felt so well relieved that the two of them had been together for the first time as the school


Setting `pad_token_id` to 50256 (first `eos_token_id`) to generate sequence


Mr. Dursley was the director of a firm called Whispers on the Isle of Wight, which he used to have a reputation for. In the end, he helped organise the boarding and boarding of the first class in the school. 
The Dursleys had everything they wanted, but they had never dreamed of being better,” she said. Then she called out one of the wizards, telling him that he hated Dumbledore and that the wizarding class, who loved him. And


OUTPUT
```
Mr. and Mrs. Dursley, of number four, Privet Drive, were proud to say that they were perfectly prepared all evening, and they felt so well relieved that the two of them had been together for the first time as the school
Mr. Dursley was the director of a firm called Whispers on the Isle of Wight, which he used to have a reputation for. In the end, he helped organise the boarding and boarding of the first class in the school. 
The Dursleys had everything they wanted, but they had never dreamed of being better,” she said. Then she called out one of the wizards, telling him that he hated Dumbledore and that the wizarding class, who loved him. And
```
The sentences sort of make sense, but there are some places such as _the boarding and boarding_ where the model outputs the same noun twice. This might be fixed if I trained the model a bit more.

# Exercise 5
Some notes:
- Might be a bit harder to fill out complete recipies that make sense, since there has to be some sort of context throughout the text (:
- The outputs of the models on the website are small, so I will have to find a way to generate larger outputs. 
- Similar projects:
    - https://huggingface.co/flax-community/t5-recipe-generation
    - https://huggingface.co/mbien/recipenlg
- Might be a bit hard to evaluate my model, but the t5 recipe generation page had a section about how they evaluated their model which I could use 